### Diagnostic Feature Audit — BUTTER-E (exploratory only)

**Not part of the final a priori feature set.** Purpose: understand what's actually driving BUTTER-E's energy variance, using every plausibly relevant raw column as a candidate — including columns that can't legitimately be used in the real model (`run_time`, known only after training finishes).

**Candidates beyond the current feature set:**
- `dataset` (one-hot, 12 categories) — which of BUTTER-E's 12 PMLB training datasets. **Note up front: unlike `run_time`, this one *is* knowable a priori** — which dataset you're about to train on is a config choice made before training starts, not an outcome. If it turns out to matter a lot, that's a legitimate candidate for the real feature set, not just this diagnostic.
- `optimizer` (one-hot) — BUTTER-E's `optimizer` column is `"Adam"` for all 37,055 rows (confirmed below), so this one-hot column carries no variance and is included only for completeness.
- `node`/`host_name` (target-encoded) — tests whether specific physical nodes systematically run hotter/cooler (shared-HPC-cluster contention). One-hot over 1,495 distinct nodes (median 11 rows/node, some with just 1) would create extremely sparse, unreliable columns, so this uses **smoothed target encoding** instead: each node's encoded value is a weighted blend of its own training-fold mean log-target and the global training-fold mean, weighted toward the global mean for nodes with few samples (smoothing constant `k=10`). Critically, **the encoding is fit on the training split only** and applied to the test split (unseen-node fallback = training global mean) — computing it on the full dataset before splitting would leak the test target into a feature that's specifically testing "does this node's own history predict its own future rows," which would make this whole diagnostic circular. Unlike `dataset`, **node identity is *not* knowable/choosable a priori** in a shared HPC/SLURM setting — you don't pick which physical node your job lands on before submitting it — so even if this matters a lot, it can't become a real feature; it's a ceiling on how well any specification-based model could ever do.

**The actual diagnostic:** train Random Forest (log1p target) on this full candidate set, then again with `run_time` added on top. If adding `run_time` causes a large jump, the dominant unexplained variance is duration-related. If it barely helps, look elsewhere (e.g. power variability itself).

In [ ]:
# IMPORTS & LOAD RAW DATA

import sys

import numpy as np
import pandas as pd
from scipy.stats import kendalltau
from sklearn.metrics import mean_absolute_percentage_error, r2_score
from sklearn.model_selection import train_test_split

sys.path.insert(0, "../../")
from src.models import random_forest

RAW = "../../data/raw/butter_e/"
df = pd.read_csv(RAW + "runs_with_standardized_energy.csv")
assert (~df["filter"]).all()

print("optimizer unique values:", df["optimizer"].unique())
print("unique nodes:", df["node"].nunique(), " | rows per node:", df.groupby("node").size().describe()[["mean", "50%", "min", "max"]].to_dict())

In [ ]:
# BUILD CANDIDATE FEATURE SET (everything except node's target encoding, added after split)

base = pd.DataFrame({
    "params": df["size"],
    "depth": df["depth"],
    "flops": 2 * df["size"],
    "epochs": 3000,
    "batch_size": df["batch_size"],
    "is_gpu": df["is_gpu"],
    "shape": df["shape"],
    "dataset": df["dataset"],
    "optimizer": df["optimizer"],
    "node": df["node"],
    "run_time": df["run_time"],
    "target": df["std_energy"],
})

shape_dummies = pd.get_dummies(base["shape"], prefix="shape").astype(int)
dataset_dummies = pd.get_dummies(base["dataset"], prefix="dataset").astype(int)
optimizer_dummies = pd.get_dummies(base["optimizer"], prefix="optimizer").astype(int)

X_full = pd.concat(
    [base[["params", "depth", "flops", "epochs", "batch_size", "is_gpu"]], shape_dummies, dataset_dummies, optimizer_dummies],
    axis=1,
)
X_full["node_raw"] = base["node"]  # kept aside for leakage-safe target encoding, not a model input itself
X_full["run_time"] = base["run_time"]

y = base["target"]
y_log = np.log1p(y)

X_full.shape

In [ ]:
# SPLIT FIRST, THEN LEAKAGE-SAFE SMOOTHED TARGET ENCODING FOR node (fit on train fold only)

idx_train, idx_test = train_test_split(X_full.index, test_size=0.2, random_state=42)

K = 10  # smoothing strength: nodes with few training rows get pulled toward the global mean
train_log = y_log.loc[idx_train]
train_node = X_full.loc[idx_train, "node_raw"]
global_mean = train_log.mean()

node_stats = train_log.groupby(train_node).agg(["mean", "count"])
node_smoothed = (node_stats["mean"] * node_stats["count"] + global_mean * K) / (node_stats["count"] + K)

# unseen-in-training nodes (can happen for rare nodes) fall back to the training global mean
X_full["node_target_enc"] = X_full["node_raw"].map(node_smoothed).fillna(global_mean)

FEATURES_NO_RUNTIME = [c for c in X_full.columns if c not in ("node_raw", "run_time")]
FEATURES_WITH_RUNTIME = FEATURES_NO_RUNTIME + ["run_time"]

print("features (no run_time):", len(FEATURES_NO_RUNTIME))
print("features (with run_time):", len(FEATURES_WITH_RUNTIME))

In [ ]:
# EVALUATION HELPER — RF, log1p target, same split throughout

def evaluate(features, label):
    X = X_full[features]
    X_train, X_test = X.loc[idx_train], X.loc[idx_test]
    y_train_log, y_test_log = y_log.loc[idx_train], y_log.loc[idx_test]
    y_test_raw = y.loc[idx_test]

    model = random_forest.build_model()
    model.fit(X_train, y_train_log)
    preds_log = model.predict(X_test)
    preds_raw = np.expm1(preds_log)

    mape_log = mean_absolute_percentage_error(y_test_log, preds_log)
    r2_log = r2_score(y_test_log, preds_log)
    mape_raw = mean_absolute_percentage_error(y_test_raw, preds_raw)
    r2_raw = r2_score(y_test_raw, preds_raw)
    tau, tau_p = kendalltau(y_test_raw, preds_raw)

    importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)

    print(f"=== {label} ===")
    print(f"n features: {len(features)}")
    print(f"MAPE (log): {mape_log:.4f}   R2 (log): {r2_log:.4f}")
    print(f"MAPE (raw): {mape_raw:.4f}   R2 (raw): {r2_raw:.4f}")
    print(f"Kendall-Tau: {tau:.4f}  (p={tau_p:.2e})")
    print("\ntop 15 feature importances:")
    print(importances.head(15))
    print()

    return dict(label=label, mape_log=mape_log, r2_log=r2_log, mape_raw=mape_raw, r2_raw=r2_raw, tau=tau, importances=importances)

In [ ]:
# RUN BOTH VERSIONS

res_no_rt = evaluate(FEATURES_NO_RUNTIME, "Full features, NO run_time")
res_with_rt = evaluate(FEATURES_WITH_RUNTIME, "Full features, WITH run_time")

In [ ]:
# SIDE-BY-SIDE SUMMARY

summary = pd.DataFrame([
    {"model": r["label"], "MAPE_raw": r["mape_raw"], "R2_raw": r["r2_raw"], "R2_log": r["r2_log"], "KendallTau": r["tau"]}
    for r in (res_no_rt, res_with_rt)
])
summary

**Result** (executed once already; re-run in VS Code to attach outputs):

| model | MAPE (raw) | R² (raw) | R² (log) | Kendall-Tau |
|---|---:|---:|---:|---:|
| Full features, no `run_time` | 0.099 | 0.969 | 0.983 | 0.933 |
| Full features, with `run_time` | 0.031 | 0.998 | 0.999 | 0.980 |

**For context — every prior BUTTER-E result on this same split:**

| feature set | MAPE (raw) | R² (raw) | Kendall-Tau |
|---|---:|---:|---:|
| 5 core features | 1.290 | 0.110 | 0.274 |
| + `is_gpu` + `shape` | 1.128 | 0.075 | 0.319 |
| + `memory_fit_ratio` | 1.150 | 0.068 | 0.314 |
| **+ `dataset` + `node_target_enc` (this notebook, no `run_time`)** | **0.099** | **0.969** | **0.933** |

**`dataset` and `node_target_enc` are the real missing signal — bigger than everything tried so far, combined, by a wide margin.** Top importances for the no-`run_time` model: `dataset_sleep` (0.160), `node_target_enc` (0.159), `dataset_connect_4` (0.117), `dataset_mnist` (0.107), `params` (0.104), `flops` (0.103), `dataset_adult` (0.095) — the `dataset_*` columns collectively dominate, and `node_target_enc` alone is the single second-most-important feature. `is_gpu`, the standout feature from the earlier diagnostic, drops to 0.0015 importance here — it wasn't wrong to matter, but `dataset` and `node_target_enc` apparently captured more of what `is_gpu` was a partial proxy for (which dataset a job trains on, and which node/hardware generation it lands on, both correlate somewhat with GPU vs. CPU assignment in this HPC scheduling data).

**The `run_time` ablation answers the original question directly: it's overwhelmingly duration-related, but the duration itself is now explained rather than mysterious.** Adding `run_time` pushes `R²` to 0.998 and MAPE to 3.1%, with `run_time` alone carrying 99.3% of feature importance — mechanically expected, since `energy ≈ power × time` and power only varies ~2.5x (235–580 W) while `run_time` varies ~590x (114s–67,632s) across BUTTER-E, so duration dominates energy variance almost by construction. That confirms the "unexplained variance is duration-related" half of the hypothesis. But the no-`run_time` model's jump from R²=0.11 to R²=0.97 just from adding `dataset` + `node_target_enc` shows *why* duration was previously unpredictable: **not primarily unmeasurable cluster contention, but a specific, legitimately-knowable variable (`dataset`) that was simply missing.** Different PMLB datasets have different sizes/complexity, which mechanically changes per-epoch compute time — that's not a mystery once it's in the model.

**Practical implications, split by whether the signal is usable a priori:**
- **`dataset` should be promoted out of this exploratory notebook into the real BUTTER-E feature set.** It's knowable before training starts (a config choice, not an outcome, same status as `is_gpu`/`shape`) and it's the single largest driver found in this entire investigation. This is the clear next action, not just a diagnostic footnote.
- **`node_target_enc` cannot become a real feature** — which physical node a SLURM job lands on isn't chosen or known in advance — but its importance (0.159, on par with the entire `dataset` block) confirms a genuine cluster-contention/hardware-heterogeneity effect exists and is non-trivial. This is a legitimate ceiling on how well any purely specification-based a priori model can predict BUTTER-E's energy, worth stating explicitly as a limitation in the thesis rather than something more feature engineering will fix.